In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("DOHMH_New_York_City_Restaurant_Inspection_Results_20260524.csv")

In [3]:
df.head()

,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,...,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,50174500,WOOFBOX,Brooklyn,214,GRAND STREET,11211.0,3474812554,NaN,01/01/1900,NaN,...,NaN,40.714003,-73.960115,301.0,34.0,55100.0,3062798.0,3.023930e+09,BK73,POINT (-73.960114971936 40.714002827832)
1,50183477,LITTLE PLAZA PIZZA,Brooklyn,188,PARKSIDE AVENUE,11226.0,6463262521,NaN,01/01/1900,NaN,...,NaN,40.655070,-73.961347,314.0,40.0,50803.0,3115925.0,3.050540e+09,BK42,POINT (-73.961346876328 40.655070193492)
2,50176286,AUNTIE ANNE'S / CINNABON,Brooklyn,625,ATLANTIC AVENUE,11217.0,7863690471,NaN,01/01/1900,NaN,...,NaN,40.683450,-73.975616,302.0,35.0,3500.0,3057470.0,3.020020e+09,BK68,POINT (-73.975615594405 40.683449625587)
3,50178605,Cluck N Moo nyc,Manhattan,1636,SAINT NICHOLAS AVENUE,10040.0,6463273058,NaN,01/01/1900,NaN,...,NaN,40.856029,-73.928807,112.0,10.0,27700.0,1063867.0,1.021610e+09,MN35,POINT (-73.928807126081 40.856028600477)
4,50184102,HAPPYGIRL LLC,0,NaN,NaN,NaN,6467034552,NaN,01/01/1900,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.shape

(295966, 27)

In [5]:
df.columns

Index(['CAMIS', 'DBA', 'BORO', 'BUILDING', 'STREET', 'ZIPCODE', 'PHONE',
       'CUISINE DESCRIPTION', 'INSPECTION DATE', 'ACTION', 'VIOLATION CODE',
       'VIOLATION DESCRIPTION', 'CRITICAL FLAG', 'SCORE', 'GRADE',
       'GRADE DATE', 'RECORD DATE', 'INSPECTION TYPE', 'Latitude', 'Longitude',
       'Community Board', 'Council District', 'Census Tract', 'BIN', 'BBL',
       'NTA', 'Location'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 295966 entries, 0 to 295965
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  295966 non-null  int64  
 1   DBA                    295964 non-null  str    
 2   BORO                   295966 non-null  str    
 3   BUILDING               294987 non-null  str    
 4   STREET                 295946 non-null  str    
 5   ZIPCODE                292879 non-null  float64
 6   PHONE                  295878 non-null  str    
 7   CUISINE DESCRIPTION    292436 non-null  str    
 8   INSPECTION DATE        295966 non-null  str    
 9   ACTION                 292528 non-null  str    
 10  VIOLATION CODE         290082 non-null  str    
 11  VIOLATION DESCRIPTION  290082 non-null  str    
 12  CRITICAL FLAG          295966 non-null  str    
 13  SCORE                  278915 non-null  float64
 14  GRADE                  145980 non-null  str    

In [7]:
df["INSPECTION DATE"] = pd.to_datetime(df["INSPECTION DATE"])

In [8]:
df = df[df["INSPECTION DATE"] != "1900-01-01"]

In [9]:
df.shape

(292528, 27)

In [10]:
df[["SCORE", "GRADE"]].isnull().sum()

SCORE     13613
GRADE    146548
dtype: int64

In [11]:
df.groupby("CAMIS").size().sort_values(ascending=False).head()

CAMIS
50138270    89
50111296    87
50139259    84
50001215    81
40365904    74
dtype: int64

In [12]:
inspection_level = (
    df.groupby([
        "CAMIS",
        "DBA",
        "BORO",
        "ZIPCODE",
        "CUISINE DESCRIPTION",
        "INSPECTION DATE"
    ], as_index=False)
    .agg(
        score=("SCORE", "max"),
        grade=("GRADE", "first"),
        violation_count=("VIOLATION CODE", "count"),
        critical_violations=("CRITICAL FLAG", lambda x: (x == "Critical").sum()),
        latitude=("Latitude", "first"),
        longitude=("Longitude", "first")
    )
)

In [13]:
inspection_level.head()

,CAMIS,DBA,BORO,ZIPCODE,CUISINE DESCRIPTION,INSPECTION DATE,score,grade,violation_count,critical_violations,latitude,longitude
0,30075445,MORRIS PARK BAKE SHOP,Bronx,10462.0,Bakery Products/Desserts,2023-08-01,38.0,NaN,3,2,40.848231,-73.855972
1,30075445,MORRIS PARK BAKE SHOP,Bronx,10462.0,Bakery Products/Desserts,2023-08-22,12.0,A,3,1,40.848231,-73.855972
2,30075445,MORRIS PARK BAKE SHOP,Bronx,10462.0,Bakery Products/Desserts,2024-11-08,10.0,A,3,1,40.848231,-73.855972
3,30075445,MORRIS PARK BAKE SHOP,Bronx,10462.0,Bakery Products/Desserts,2026-02-27,7.0,A,2,1,40.848231,-73.855972
4,30191841,D.J. REYNOLDS,Manhattan,10019.0,Irish,2023-04-23,10.0,A,2,2,40.767326,-73.984310


In [14]:
inspection_level.shape

(83512, 12)

In [15]:
inspection_level.to_csv("inspection_level_clean.csv", index=False)

In [16]:
county_df = pd.read_csv("QOL(County Level).csv", encoding="latin1")
school_df = pd.read_csv("QOL(Public School Level).csv", encoding="latin1")

/tmp/ipykernel_390/4254205969.py:2: DtypeWarning: Columns (0: SCH_NAME, 1: LSTREET1, 2: LCITY, 3: cityhelper, 4: countyhelper, 5: LSTATE, 6: NMCNTY, 7: ULOCALE, 8: 2016 Crime Rate, 9: Unemployment, 10: 2020PopulrVoteParty, 11: 2020 PopulrMajor%, 12: AQI%Good, 13: %CvgCityPark, 14: %CvgStatePark, 15: Cost of Living, 16: 2022 Median Income, 17: AVG C2I, 18: 1p0c, 19: 1p1c, 20: 1p2c, 21: 1p3c, 22: 1p4c, 23: 2p0c, 24: 2p1c, 25: 2p2c, 26: 2p3c, 27: 2p4c, 28: SCHOOL_LEVEL, 29: SCHOOL_TYPE, 30: GSLO, 31: GSHI, 32: Unnamed: 44, 33: Unnamed: 45, 34: Unnamed: 46, 35: Unnamed: 47) have mixed types. Specify dtype option on import or set low_memory=False.
  school_df = pd.read_csv("QOL(Public School Level).csv", encoding="latin1")


In [17]:
county_df.head()

,countyhelper,LSTATE,NMCNTY,FIPS,LZIP,ULOCALE,Overall Rank,2022 Population,2016 Crime Rate,Unemployment,...,1p3c,1p4c,2p0c,2p1c,2p2c,2p3c,2p4c,Stu:Tea Rank,Diversity Rank (Race),Diversity Rank (Gender)
0,VACharles City County,VA,Charles City County,51036,23030,42-Rural: Distant,NaN,"6,605",8/1000,3.21%,...,111.16%,119.90%,67.97%,90.07%,105.57%,123.82%,131.87%,135,1,25
1,TXMcmullen County,TX,McMullen County,48311,78072,43-Rural: Remote,NaN,576,47/1000,1.81%,...,105.46%,111.95%,72.03%,90.73%,104.21%,120.05%,127.11%,3,2,87
2,TXTerrell County,TX,Terrell County,48443,79848,43-Rural: Remote,NaN,693,20/1000,3.54%,...,127.10%,135.84%,87.96%,110.73%,125.11%,145.91%,153.79%,12,3,47
3,AKSkagway Municipality,AK,Skagway Municipality,2230,99840,43-Rural: Remote,NaN,"1,081",13/1000,7.19%,...,121.39%,128.32%,69.18%,94.01%,113.02%,132.18%,139.30%,15,4,9
4,GABaker County,GA,Baker County,13007,39870,42-Rural: Distant,NaN,"2,788",0,4.19%,...,122.57%,131.91%,85.54%,108.73%,124.45%,141.99%,153.63%,26,5,60


In [18]:
county_df.columns

Index(['countyhelper', 'LSTATE', 'NMCNTY', 'FIPS', 'LZIP', 'ULOCALE',
       'Overall Rank', '2022 Population', '2016 Crime Rate', 'Unemployment',
       '2020PopulrVoteParty', '2020 PopulrMajor%', 'AQI%Good',
       'WaterQualityVPV', 'ParkScore2023 Rank', '%CvgCityPark', 'NtnlPrkCnt',
       '%CvgStatePark', 'Cost of Living', '2022 Median Income', 'AVG C2I',
       '1p0c', '1p1c', '1p2c', '1p3c', '1p4c', '2p0c', '2p1c', '2p2c', '2p3c',
       '2p4c', 'Stu:Tea Rank', 'Diversity Rank (Race)',
       'Diversity Rank (Gender)'],
      dtype='str')

In [19]:
county_df.shape

(3134, 34)

In [20]:
county_df["LZIP"].nunique()

3114

In [21]:
school_df.head()

,SCH_NAME,LSTREET1,LCITY,cityhelper,countyhelper,LSTATE,NMCNTY,FIPS,LZIP,ULOCALE,...,SCHOOL_TYPE,GSLO,GSHI,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47
0,SATELLITE ACADEMY HIGH SCHOOL,120 W 30TH ST,NEW YORK,NYNew York,NYNew York County,NY,New York County,36061.0,10001.0,11-City: Large,...,Regular school,09,12,NaN,NaN,NaN,Row Labels,Average of Stu:Tea Rank,Sum of Diversity Rank (Race),Sum of Diversity Rank (Gender)
1,PS 33 CHELSEA PREP,281 9TH AVE,NEW YORK,NYNew York,NYNew York County,NY,New York County,36061.0,10001.0,11-City: Large,...,Regular school,PK,05,NaN,NaN,NaN,AKAleutians East Borough,7013,195178,108085
2,GIRLS PREPARATORY CHARTER SCHOOL OF NEW YORK,442 E HOUSTON ST-RM 312,NEW YORK,NYNew York,NYNew York County,NY,New York County,36061.0,10002.0,11-City: Large,...,Regular school,PK,08,NaN,NaN,NaN,AKAleutians West Census Area,27438.33333,350869,214044
3,NEW DESIGN HIGH SCHOOL,350 GRAND ST,NEW YORK,NYNew York,NYNew York County,NY,New York County,36061.0,10002.0,11-City: Large,...,Regular school,09,12,NaN,NaN,NaN,AKAnchorage Municipality,46710.64286,680329,4483885
4,LOWER MANHATTAN ARTS ACADEMY,350 GRAND ST,NEW YORK,NYNew York,NYNew York County,NY,New York County,36061.0,10002.0,11-City: Large,...,Regular school,09,12,NaN,NaN,NaN,AKBethel Census Area,43854.14634,3531560,1673318


In [22]:
school_df.columns

Index(['SCH_NAME', 'LSTREET1', 'LCITY', 'cityhelper', 'countyhelper', 'LSTATE',
       'NMCNTY', 'FIPS', 'LZIP', 'ULOCALE', '2022 Population',
       '2016 Crime Rate', 'Unemployment', '2020PopulrVoteParty',
       '2020 PopulrMajor%', 'AQI%Good', 'WaterQualityVPV',
       'ParkScore2023 Rank', '%CvgCityPark', 'NtnlPrkCnt', '%CvgStatePark',
       'Cost of Living', '2022 Median Income', 'AVG C2I', '1p0c', '1p1c',
       '1p2c', '1p3c', '1p4c', '2p0c', '2p1c', '2p2c', '2p3c', '2p4c',
       'Stu:Tea Rank', 'Diversity Rank (Race)', 'Diversity Rank (Gender)',
       'SCHOOL_LEVEL', 'SCHOOL_TYPE', 'GSLO', 'GSHI', 'Unnamed: 41',
       'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45',
       'Unnamed: 46', 'Unnamed: 47'],
      dtype='str')

In [23]:
school_df.shape

(100722, 48)

In [24]:
school_df["LZIP"].nunique()

20928

In [25]:
income_df = county_df[[
    "LZIP",
    "2022 Median Income",
    "Unemployment",
    "2016 Crime Rate",
    "Cost of Living"
]]

In [26]:
income_df = income_df.rename(columns={
    "LZIP": "ZIPCODE",
    "2022 Median Income": "median_income",
    "Unemployment": "unemployment_rate",
    "2016 Crime Rate": "crime_rate",
    "Cost of Living": "cost_of_living"
})

In [27]:
income_df.head()

,ZIPCODE,median_income,unemployment_rate,crime_rate,cost_of_living
0,23030,"$78,038.78",3.21%,8/1000,"$75,531.37"
1,78072,"$67,513.81",1.81%,47/1000,"$63,913.28"
2,79848,"$55,946.62",3.54%,20/1000,"$64,361.02"
3,99840,"$85,446.30",7.19%,13/1000,"$87,709.32"
4,39870,"$52,946.23",4.19%,0,"$59,389.29"


In [28]:
inspection_level["ZIPCODE"] = (
    inspection_level["ZIPCODE"]
    .astype(str)
    .str[:5]
)

income_df["ZIPCODE"] = (
    income_df["ZIPCODE"]
    .astype(str)
    .str[:5]
)

In [29]:
merged_df = inspection_level.merge(
    income_df,
    on="ZIPCODE",
    how="left"
)

In [30]:
merged_df.head()

,CAMIS,DBA,BORO,ZIPCODE,CUISINE DESCRIPTION,INSPECTION DATE,score,grade,violation_count,critical_violations,latitude,longitude,median_income,unemployment_rate,crime_rate,cost_of_living
0,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2023-08-01,38.0,NaN,3,2,40.848231,-73.855972,NaN,NaN,NaN,NaN
1,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2023-08-22,12.0,A,3,1,40.848231,-73.855972,NaN,NaN,NaN,NaN
2,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2024-11-08,10.0,A,3,1,40.848231,-73.855972,NaN,NaN,NaN,NaN
3,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2026-02-27,7.0,A,2,1,40.848231,-73.855972,NaN,NaN,NaN,NaN
4,30191841,D.J. REYNOLDS,Manhattan,10019,Irish,2023-04-23,10.0,A,2,2,40.767326,-73.984310,NaN,NaN,NaN,NaN


In [31]:
inspection_level["ZIPCODE"].head()

0    10462
1    10462
2    10462
3    10462
4    10019
Name: ZIPCODE, dtype: str

In [32]:
income_df["ZIPCODE"].head()

0    23030
1    78072
2    79848
3    99840
4    39870
Name: ZIPCODE, dtype: str

In [33]:
inspection_level["ZIPCODE"].dtype

<StringDtype(na_value=nan)>

In [34]:
income_df["ZIPCODE"].dtype

<StringDtype(na_value=nan)>

In [35]:
income_df[income_df["ZIPCODE"].isin(["10001", "10002", "10003", "10462", "11201", "11375"])]

,ZIPCODE,median_income,unemployment_rate,crime_rate,cost_of_living


In [36]:
income_df[income_df["ZIPCODE"].str.startswith(("100", "101", "102", "103", "104", "111", "112", "113", "114", "116"))].head(20)

,ZIPCODE,median_income,unemployment_rate,crime_rate,cost_of_living
2794,10305,"$103,213.48",5.82%,20/1000,"$118,032.58"
3093,10456,"$48,566.82",8.45%,22/1000,"$102,815.48"
3099,10463,"$112,986.80",4.80%,21/1000,"$137,874.70"
3100,11364,"$79,063.55",5.51%,21/1000,"$124,627.90"
3112,11208,"$70,138.38",6.24%,21/1000,"$115,425.65"


In [37]:
merged_df["median_income"].isna().mean()

np.float64(0.9823258932847974)

County-level dataset produced insufficient ZIP matches (~98% missing after merge), so project pivoted to school-level ZIP dataset for better NYC ZIP matches. 

In [38]:
zip_demographics_df = school_df[[
    "LZIP",
    "2022 Median Income",
    "Unemployment",
    "2016 Crime Rate",
    "Cost of Living"
]].copy()

In [39]:
zip_demographics_df = zip_demographics_df.rename(columns={
    "LZIP": "ZIPCODE",
    "2022 Median Income": "median_income",
    "Unemployment": "unemployment_rate",
    "2016 Crime Rate": "crime_rate",
    "Cost of Living": "cost_of_living"
})

In [40]:
zip_demographics_df = zip_demographics_df.dropna(subset=["ZIPCODE"])

In [41]:
zip_demographics_df["ZIPCODE"] = (
    zip_demographics_df["ZIPCODE"]
    .astype(float)
    .astype(int)
    .astype(str)
)

In [42]:
zip_demographics_df = zip_demographics_df.drop_duplicates(subset="ZIPCODE")

In [43]:
zip_demographics_df.shape

(20928, 5)

In [44]:
zip_demographics_df.head()

,ZIPCODE,median_income,unemployment_rate,crime_rate,cost_of_living
0,10001,"$112,986.80",4.80%,25/1000,"$137,874.70"
2,10002,"$112,986.80",4.80%,25/1000,"$137,874.70"
33,10003,"$112,986.80",4.80%,25/1000,"$137,874.70"
49,10004,"$112,986.80",4.80%,5/1000,"$137,874.70"
55,10006,"$112,986.80",4.80%,25/1000,"$137,874.70"


In [45]:
merged_df = inspection_level.merge(
    zip_demographics_df,
    on="ZIPCODE",
    how="left"
)

In [46]:
merged_df["median_income"].isna().mean()

np.float64(0.06620605421975286)

In [47]:
merged_df[["ZIPCODE", "median_income", "unemployment_rate", "crime_rate", "cost_of_living"]].head(20)

,ZIPCODE,median_income,unemployment_rate,crime_rate,cost_of_living
0,10462,"$48,566.82",8.45%,27/1000,"$102,815.48"
1,10462,"$48,566.82",8.45%,27/1000,"$102,815.48"
2,10462,"$48,566.82",8.45%,27/1000,"$102,815.48"
3,10462,"$48,566.82",8.45%,27/1000,"$102,815.48"
4,10019,"$112,986.80",4.80%,10/1000,"$137,874.70"
5,10019,"$112,986.80",4.80%,10/1000,"$137,874.70"
6,10019,"$112,986.80",4.80%,10/1000,"$137,874.70"
7,11224,"$70,138.38",6.24%,21/1000,"$115,425.65"
8,11224,"$70,138.38",6.24%,21/1000,"$115,425.65"
9,11234,"$70,138.38",6.24%,21/1000,"$115,425.65"


In [48]:
merged_df["median_income"] = (
    merged_df["median_income"]
    .replace(r"[\$,]", "", regex=True)
    .astype(float)
)

merged_df["unemployment_rate"] = (
    merged_df["unemployment_rate"]
    .str.replace("%", "")
    .astype(float)
)

merged_df["crime_rate"] = (
    merged_df["crime_rate"]
    .str.replace("/1000", "")
    .astype(float)
)

merged_df["cost_of_living"] = (
    merged_df["cost_of_living"]
    .replace(r"[\$,]", "", regex=True)
    .astype(float)
)

In [49]:
merged_df[[
    "median_income",
    "unemployment_rate",
    "crime_rate",
    "cost_of_living"
]].dtypes

median_income        float64
unemployment_rate    float64
crime_rate           float64
cost_of_living       float64
dtype: object

In [50]:
merged_df.head()

,CAMIS,DBA,BORO,ZIPCODE,CUISINE DESCRIPTION,INSPECTION DATE,score,grade,violation_count,critical_violations,latitude,longitude,median_income,unemployment_rate,crime_rate,cost_of_living
0,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2023-08-01,38.0,NaN,3,2,40.848231,-73.855972,48566.82,8.45,27.0,102815.48
1,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2023-08-22,12.0,A,3,1,40.848231,-73.855972,48566.82,8.45,27.0,102815.48
2,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2024-11-08,10.0,A,3,1,40.848231,-73.855972,48566.82,8.45,27.0,102815.48
3,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2026-02-27,7.0,A,2,1,40.848231,-73.855972,48566.82,8.45,27.0,102815.48
4,30191841,D.J. REYNOLDS,Manhattan,10019,Irish,2023-04-23,10.0,A,2,2,40.767326,-73.984310,112986.80,4.80,10.0,137874.70


In [51]:
merged_df.columns

Index(['CAMIS', 'DBA', 'BORO', 'ZIPCODE', 'CUISINE DESCRIPTION',
       'INSPECTION DATE', 'score', 'grade', 'violation_count',
       'critical_violations', 'latitude', 'longitude', 'median_income',
       'unemployment_rate', 'crime_rate', 'cost_of_living'],
      dtype='str')

In [52]:
merged_df.describe()

,CAMIS,INSPECTION DATE,score,violation_count,critical_violations,latitude,longitude,median_income,unemployment_rate,crime_rate,cost_of_living
count,8.351200e+04,83512,82483.000000,83512.000000,83512.000000,82937.000000,82937.000000,77983.000000,77983.000000,77983.000000,77983.000000
mean,4.798718e+07,2024-08-05 01:02:09.667592,17.815926,3.437398,1.844214,40.727937,-73.942622,86367.039198,5.760921,19.770732,124324.265511
min,3.007544e+07,2009-05-22 00:00:00,0.000000,0.000000,0.000000,40.499563,-74.248708,48566.820000,2.960000,4.000000,102815.480000
25%,5.000222e+07,2023-10-03 00:00:00,9.000000,2.000000,1.000000,40.688083,-73.989340,70138.380000,4.800000,21.000000,115425.650000
50%,5.008937e+07,2024-10-07 00:00:00,13.000000,3.000000,1.000000,40.732743,-73.958395,79063.550000,5.510000,21.000000,124627.900000
75%,5.012910e+07,2025-08-12 00:00:00,24.000000,4.000000,3.000000,40.761375,-73.901756,112986.800000,6.240000,25.000000,137874.700000
max,5.018639e+07,2026-05-21 00:00:00,214.000000,24.000000,13.000000,40.912822,-73.700928,139281.770000,8.450000,27.000000,137874.700000
std,3.802623e+06,NaN,14.838014,2.215122,1.510855,0.068001,0.075748,21993.595759,1.055983,6.106509,11496.389118


In [53]:
merged_df.groupby("BORO")["score"].mean().sort_values()

BORO
Staten Island    17.014031
Manhattan        17.088427
Bronx            17.487159
Brooklyn         17.874357
0                17.962963
Queens           19.175304
Name: score, dtype: float64

In [54]:
merged_df[[
    "score",
    "critical_violations",
    "median_income",
    "unemployment_rate",
    "crime_rate"
]].corr()

,score,critical_violations,median_income,unemployment_rate,crime_rate
score,1.000000,0.852887,-0.019949,-0.001328,0.016596
critical_violations,0.852887,1.000000,-0.017328,-0.002436,0.018467
median_income,-0.019949,-0.017328,1.000000,-0.887176,-0.170931
unemployment_rate,-0.001328,-0.002436,-0.887176,1.000000,0.064544
crime_rate,0.016596,0.018467,-0.170931,0.064544,1.000000


In [55]:
merged_df.to_csv("final_restaurant_dataset.csv", index=False)

In [56]:
merged_df["grade"].value_counts(dropna=False)

grade
A      44651
NaN    28342
B       4402
C       2306
N       2197
Z       1024
P        590
Name: count, dtype: int64

In [57]:
merged_df[merged_df["grade"].isna()][["score"]].describe()

,score
count,27317.000000
mean,28.713402
std,17.293957
min,0.000000
25%,19.000000
50%,26.000000
75%,35.000000
max,191.000000


Adding new zip-code census income data.
The original supplementary dataset that contained info about income data and demographic statistics, only had median income estimates grouped by borough. We want to make our analysis more granular and therefore are adding this new data set from census that has the median income level for each zipcode within every neighborhood in NYC. 

In [59]:
api_key = "a60926e6883beb453ca71fb7a2d70f0f489554d5"

url = f"https://api.census.gov/data/2023/acs/acs5?get=B19013_001E,NAME&for=zip%20code%20tabulation%20area:*&key={api_key}"

acs_income = pd.read_json(url)

# first row becomes headers
acs_income.columns = acs_income.iloc[0]
# remove duplicated header row
acs_income = acs_income.iloc[1:]

# rename columns
acs_income = acs_income.rename(columns={
    "zip code tabulation area": "ZIPCODE",
    "B19013_001E": "median_income"
})

# keep only the columns we want to add 
acs_income = acs_income[["ZIPCODE", "median_income"]]

acs_income.head()

,ZIPCODE,median_income
1,00601,18571
2,00602,21702
3,00603,19243
4,00606,20226
5,00610,23732


In [60]:
merged_df = merged_df.drop(columns=["median_income"], errors="ignore")

merged_df = merged_df.merge(
    acs_income,
    on="ZIPCODE",
    how="left"
)

merged_df[["ZIPCODE", "median_income"]].head(20)

,ZIPCODE,median_income
0,10462,63460
1,10462,63460
2,10462,63460
3,10462,63460
4,10019,121835
5,10019,121835
6,10019,121835
7,11224,41449
8,11224,41449
9,11234,94434


In [61]:
merged_df.head()

,CAMIS,DBA,BORO,ZIPCODE,CUISINE DESCRIPTION,INSPECTION DATE,score,grade,violation_count,critical_violations,latitude,longitude,unemployment_rate,crime_rate,cost_of_living,median_income
0,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2023-08-01,38.0,NaN,3,2,40.848231,-73.855972,8.45,27.0,102815.48,63460
1,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2023-08-22,12.0,A,3,1,40.848231,-73.855972,8.45,27.0,102815.48,63460
2,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2024-11-08,10.0,A,3,1,40.848231,-73.855972,8.45,27.0,102815.48,63460
3,30075445,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,2026-02-27,7.0,A,2,1,40.848231,-73.855972,8.45,27.0,102815.48,63460
4,30191841,D.J. REYNOLDS,Manhattan,10019,Irish,2023-04-23,10.0,A,2,2,40.767326,-73.984310,4.80,10.0,137874.70,121835


In [63]:
merged_df.to_csv("final_restaurant_dataset.csv", index=False)